# Generating Points on the Flag Manifold

## Setup

In [ ]:
import torch
torch.set_default_dtype(torch.float64)
import numpy as np 
import qmcpy as qp
import agsutil
import time
import pandas as pd
import os
import matplotlib
from matplotlib import pyplot
MPLP = agsutil.mpl_setup()
agsutil.print_data_signatures(MPLP,"MPLP",verbose_indent=0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch device = %s"%DEVICE)
icdf_normal = torch.distributions.Normal(loc=torch.zeros(1,device=DEVICE),scale=torch.ones(1,device=DEVICE)).icdf

MPLP['PW'] = 30
MPLP['FS'] = 30
MPLP['COLORS'] a list of length 10
MPLP['LINESTYLES'] a list of length 10
MPLP['MARKERS'] a list of length 12
torch device = cpu


In [56]:
def tff_qr(u, lam):
    t = lam.size(-1)
    assert u.size(-1)==(t**2)
    assert (0<=u).all()
    assert (u<=1).all()
    u = u.reshape((*u.shape[:-1],t,t))
    v = icdf_normal(u)
    q,r = torch.linalg.qr(v)
    assert torch.allclose(torch.einsum("...ij,...jk->...ik",q,r),v)
    f = torch.einsum("...ij,...j,...kj->...ik",q,lam,q)
    return f
def tff_eig(u, lam):
    t = lam.size(-1) 
    assert u.size(-1)==(t*(t+1)//2)
    alpha = icdf_normal(u[...,:t])/np.sqrt(2)
    beta = icdf_normal(u[...,t:])
    il0,il1 = torch.tril_indices(n,n,offset=-1,device=DEVICE)
    v = torch.eye(n,device=DEVICE)*alpha[...,None]
    v[...,il0,il1] = beta
    v += v.tril(-1).transpose(dim0=-2,dim1=-1)
    assert torch.allclose(v[...,torch.arange(n,device=DEVICE),torch.arange(n,device=DEVICE)],alpha)
    gamma,q = torch.linalg.eigh(v) # TODO: flip sign of q since it is only unique up to multiplying each column by {-1,1}
    assert torch.allclose(torch.einsum("...ij,...j,...kj->...ik",q,gamma,q),v)
    f = torch.einsum("...ij,...j,...kj->...ik",q,lam,q)
    return f
def genf_qr_iid_x_equal_w(n, lam, seed=None, verbose=True):
    rng = torch.Generator(device=DEVICE) if seed is None else torch.Generator(device=DEVICE).manual_seed(seed)
    t = lam.size(-1)
    u = torch.rand((n,t**2),generator=rng,device=DEVICE)
    x = tff_qr(u,lam)
    w = torch.ones(n,device=DEVICE)/n
    return x,w
def genf_qr_ld_x_equal_w(n, lam, seed=None, verbose=True):
    rng = torch.Generator(device=DEVICE) if seed is None else torch.Generator(device=DEVICE).manual_seed(seed)
    t = lam.size(-1)
    u = torch.from_numpy(qp.Halton(t**2,seed=seed,randomize="NUS",warn=False)(n)).to(DEVICE)
    x = tff_qr(u,lam)
    w = torch.ones(n,device=DEVICE)/n
    return x,w
def genf_eig_iid_x_equal_w(n, lam, seed=None, verbose=True):
    rng = torch.Generator(device=DEVICE) if seed is None else torch.Generator(device=DEVICE).manual_seed(seed)
    t = lam.size(-1)
    u = torch.rand((n,t*(t+1)//2),generator=rng,device=DEVICE)
    x = tff_eig(u,lam)
    w = torch.ones(n,device=DEVICE)/n
    return x,w
def genf_eig_ld_x_equal_w(n, lam, seed=None, verbose=True):
    rng = torch.Generator(device=DEVICE) if seed is None else torch.Generator(device=DEVICE).manual_seed(seed)
    t = lam.size(-1)
    u = torch.from_numpy(qp.Halton(t*(t+1)//2,seed=seed,randomize="NUS",warn=False)(n)).to(DEVICE)
    x = tff_eig(u,lam)
    w = torch.ones(n,device=DEVICE)/n
    return x,w

x,w = genf_qr_iid_x_equal_w(n=3,lam=torch.rand(4,generator=rng,device=DEVICE))
x,w = genf_qr_ld_x_equal_w(n=3,lam=torch.rand(4,generator=rng,device=DEVICE))
x,w = genf_eig_iid_x_equal_w(n=3,lam=torch.rand(4,generator=rng,device=DEVICE))
x,w = genf_eig_ld_x_equal_w(n=3,lam=torch.rand(4,generator=rng,device=DEVICE))